In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sbn
import statistics as sts

In [2]:
# Importar e visualizar dados.

dataset = pd.read_csv("C:/Users/mrwag/OneDrive/Big Data/DS/21.Textos/Churn.csv", sep = ';')
dataset.head()

,X0,X1,X2,X3,X4,X4.1,X6,X7,X8,X9,X10,X11
0,1,619,RS,Feminino,42,2,0,1,1,1,10134888.0,1
1,2,608,SC,Feminino,41,1,8380786,1,0,1,11254258.0,0
2,3,502,RS,Feminino,42,8,1596608,3,1,0,11393157.0,1
3,4,699,RS,Feminino,39,1,0,2,0,0,9382663.0,0
4,5,850,SC,Feminino,43,2,12551082,1,1,1,790841.0,0


In [3]:
# Nomeando colunas.

dataset.columns = ["ID", "Score", "Estado", "Genero", "Idade", "Patrimonio", "Saldo", "Produtos", "TemCartCredito",
                   "Ativo", "Salario", "Saiu"]
dataset.head()

,ID,Score,Estado,Genero,Idade,Patrimonio,Saldo,Produtos,TemCartCredito,Ativo,Salario,Saiu
0,1,619,RS,Feminino,42,2,0,1,1,1,10134888.0,1
1,2,608,SC,Feminino,41,1,8380786,1,0,1,11254258.0,0
2,3,502,RS,Feminino,42,8,1596608,3,1,0,11393157.0,1
3,4,699,RS,Feminino,39,1,0,2,0,0,9382663.0,0
4,5,850,SC,Feminino,43,2,12551082,1,1,1,790841.0,0


In [4]:
dataset.isnull().sum()

ID                0
Score             0
Estado            0
Genero            8
Idade             0
Patrimonio        0
Saldo             0
Produtos          0
TemCartCredito    0
Ativo             0
Salario           7
Saiu              0
dtype: int64

In [5]:
mediana = sts.median(dataset['Salario'])
mediana

70518.0

In [6]:
# Substituindo os valores nulos pela mediana.

dataset['Salario']= dataset['Salario'].fillna(mediana) # Atualizado


In [7]:
# Verificando se há valores nulos.

dataset['Salario'].isnull().sum()

np.int64(0)

In [8]:
grupo = dataset.groupby('Genero').size()
grupo

Genero
F              2
Fem            1
Feminino     461
M              6
Masculino    521
dtype: int64

In [9]:
dataset['Genero'].isnull().sum()

np.int64(8)

In [10]:
dataset['Genero'] = dataset['Genero'].fillna('Masculino', inplace= True) # ANTIGO. ATUAL SEM INPLACE.

C:\Users\mrwag\AppData\Local\Temp\ipykernel_11316\3506873554.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  dataset['Genero'] = dataset['Genero'].fillna('Masculino', inplace= True) # ANTIGO. ATUAL SEM INPLACE.


In [11]:
dataset['Genero'].isnull().sum()

np.int64(0)

In [12]:
# Padronizando de acordo com o domínio.

dataset.loc[dataset['Genero'] == 'M', 'Genero'] = 'Masculino'
dataset.loc[dataset['Genero'].isin(['Fem', 'F']), 'Genero'] = 'Feminino'

# Visualizando o resultado.

dataset.groupby(['Genero']).size()


Genero
Feminino     464
Masculino    535
dtype: int64

In [13]:
# Idades fora do domínio.

dataset['Idade'].describe()

count    999.000000
mean      38.902903
std       11.401912
min      -20.000000
25%       32.000000
50%       37.000000
75%       44.000000
max      140.000000
Name: Idade, dtype: float64

In [14]:
# Visualizar.

dataset.loc[(dataset['Idade'] < 0) | (dataset['Idade'] > 120)]

,ID,Score,Estado,Genero,Idade,Patrimonio,Saldo,Produtos,TemCartCredito,Ativo,Salario,Saiu
867,869,636,RS,Feminino,-10,1,17083346,1,1,0,11051028.0,1
984,986,773,RS,Masculino,-20,1,12453278,2,0,1,1172357.0,0
990,992,655,RS,Masculino,140,5,93147,2,1,0,6621413.0,0


In [15]:
# Calculando a mediana.

media = sts.median(dataset['Idade'])
media

37

In [16]:
# Substituindo.

dataset.loc[(dataset['Idade'] < 0) | (dataset['Idade'] > 120), 'Idade'] = media

In [17]:
# Verificando o resultado.

dataset.loc[(dataset['Idade'] < 0) | (dataset['Idade'] > 120)]

,ID,Score,Estado,Genero,Idade,Patrimonio,Saldo,Produtos,TemCartCredito,Ativo,Salario,Saiu


In [18]:
# Dados duplicados, buscamos pelo ID.

dataset[dataset.duplicated(['ID'], keep = False)]

,ID,Score,Estado,Genero,Idade,Patrimonio,Saldo,Produtos,TemCartCredito,Ativo,Salario,Saiu
80,81,665,RS,Feminino,34,1,9664554,2,0,0,17141366.0,0
81,81,665,RS,Feminino,34,1,9664554,2,0,0,17141366.0,0


In [19]:
# Excluindo duplicatas.

dataset.drop_duplicates(subset= 'ID', keep = 'first', inplace = True)

In [20]:
# Buscando duplicatas para ver o resultado.

dataset[dataset.duplicated(['ID'], keep = False)]

,ID,Score,Estado,Genero,Idade,Patrimonio,Saldo,Produtos,TemCartCredito,Ativo,Salario,Saiu


In [21]:
# Outliers em salário, considerando 2 desvios padrão.

desvio = sts.stdev(dataset['Salario'])
desvio

528988918.4679201

In [22]:
# Definindo padrão maior que 2 desvios padrão.

dataset.loc[dataset['Salario']>= 2 * desvio]


,ID,Score,Estado,Genero,Idade,Patrimonio,Saldo,Produtos,TemCartCredito,Ativo,Salario,Saiu
7,8,376,PR,Feminino,29,4,11504674,4,1,0,1.193469e+10,1
116,118,668,PR,Feminino,37,6,1678644,1,1,0,1.156383e+10,0
170,172,484,RS,Feminino,29,4,13011439,1,1,0,1.640179e+09,0
230,232,673,RS,Masculino,72,1,0,2,0,1,1.119812e+09,0


In [23]:
# Calculamos a median.

mediana = sts.median(dataset['Salario'])
mediana

8637195.5

In [24]:
# Atribuindo a mediana ao salário.

dataset.loc[dataset['Salario']>= 2 * desvio, 'Salario'] = mediana

In [25]:
# Checando o resultado.

dataset.loc[dataset['Salario']>= 2 * desvio]

,ID,Score,Estado,Genero,Idade,Patrimonio,Saldo,Produtos,TemCartCredito,Ativo,Salario,Saiu


In [26]:
dataset.head()

,ID,Score,Estado,Genero,Idade,Patrimonio,Saldo,Produtos,TemCartCredito,Ativo,Salario,Saiu
0,1,619,RS,Feminino,42,2,0,1,1,1,10134888.0,1
1,2,608,SC,Feminino,41,1,8380786,1,0,1,11254258.0,0
2,3,502,RS,Feminino,42,8,1596608,3,1,0,11393157.0,1
3,4,699,RS,Feminino,39,1,0,2,0,0,9382663.0,0
4,5,850,SC,Feminino,43,2,12551082,1,1,1,790841.0,0


In [27]:
dataset.shape

(998, 12)